In [ ]:
#| default_exp dataset.core

# Dataset

> This module contain the implementation of the Dataset class

## Overview

This module defines dataset utilities.
It includes functions for sampling quantum spin configurations and classical shadows,
plus the `Dataset` class for organizing quantum dataset within the qdisc pipeline.



In [ ]:
#| hide
from nbdev import show_doc
from nbdev.showdoc import *

In [ ]:
#| export

import jax
from jax import numpy as jnp
from functools import partial
from dataclasses import dataclass
from typing import Sequence

## Helper functions for data creation

These functions help build quantum training data from exact wavefunctions. The helpers include:

- `sample_spin_configurations(...)`: sample bitstring configurations from a wavefunction probability distribution.
- `get_classical_shadow(...)`: generate classical shadow measurements in X/Y/Z bases.


In [ ]:
#| export


#@jax.jit
def sample_spin_configurations(wave_function, num_samples, N, key):
    """
        samples spin config. from an exact wavefunction
        
        Args:
        
          wave_function: (2**N,) complex64
          num_samples: int, nbr of samples
          N: int, nbr of sites
          key: a JAX PRNGKey

        Returns:
        
          spin_configurations: (num_samples, N) int32, samples of spin configurations
    """
    wave_function = wave_function.reshape(-1)
    probabilities = jnp.abs(wave_function) ** 2

    # Sample indices based on the probabilities
    sampled_indices = jax.random.choice(
        key,
        a=len(probabilities),
        p=probabilities,
        shape=(num_samples,),
        replace=True
    )

    # Convert sampled indices to spin configurations
    def int_to_spin_config(idx):
        return jnp.array([(idx >> i) & 1 for i in range(N)])

    spin_configurations = jax.vmap(int_to_spin_config)(sampled_indices)

    return spin_configurations



# Single‑qubit rotations for X, Y, Z bases
sqrt2 = 1 / jnp.sqrt(2)
H     = sqrt2 * jnp.array([[1,  1],
                           [1, -1]], dtype=jnp.complex64)
S_dag = jnp.array([[1, 0],
                   [0, 1j]], dtype=jnp.complex64)


# X->0, Y->1, Z->2
ROTATIONS = jnp.stack([H,
                       S_dag @ H,
                       jnp.eye(2, dtype=jnp.complex64)], axis=0)   # shape (3,2,2)


@partial(jax.jit, static_argnums=(1,2,4))
def get_classical_shadow(psi, num_shots, N, rng_key, bases = 'XYZ'):
    """
    generate classical shadows from an exact wavefunction
    
    Args:
    
      psi:        (2**N,) complex64 from, f. ex.,  exact diagonalization
      num_shots:  int
      N:          int
      rng_key:    a JAX PRNGKey
      bases:      which measurement basis, default 'XYZ' -> {X,Y,Z} randomly, other available option: 'X', 'Y', 'Z', 'XY', 'YZ'
    
    Returns:
    
      shadows: (num_shots, 2*N) int32  interleaved [basis,outcome] pairs
               basis in {2,3,4} = {X,Y,Z}, outcome in {0,1} = {+1,-1}
    """
    # reshape psi so qubit axis = last dims
    psi_tensor = psi.reshape((2,)*N)

    # draw bases per shot: shape (num_shots, N), values in {0,1,2}
    rng_key, sub = jax.random.split(rng_key)
    if bases == 'XYZ':
      bases = jax.random.randint(sub, (num_shots, N), 0, 3, dtype=jnp.int32)
    elif bases == 'X':
      bases = jnp.zeros((num_shots, N), dtype=jnp.int32)
    elif bases == 'Y':
      bases = jnp.ones((num_shots, N), dtype=jnp.int32)
    elif bases == 'Z':
      bases = 2*jnp.ones((num_shots, N), dtype=jnp.int32)
    elif bases == 'XZ':
      bases = jax.random.randint(sub, (num_shots, N), 0, 2, dtype=jnp.int32)
      bases = bases*2
    elif bases == 'XY':
      bases = jax.random.randint(sub, (num_shots, N), 0, 2, dtype=jnp.int32)
    elif bases == 'YZ':
      bases = jax.random.randint(sub, (num_shots, N), 0, 2, dtype=jnp.int32)
      bases = bases + 1



    # keys for sampling
    keys = jax.random.split(rng_key, num_shots)

    def measure_one(basis_row, key):
        # start from psi per shot
        state = psi_tensor
        # apply each qubit rotation in turn
        for qubit in range(N):
            U = ROTATIONS[basis_row[qubit]]   # (2,2)
            # tensordot over the qubit axis
            state = jnp.tensordot(U, state, axes=[[1],[qubit]])
            # move the new axis into position qubit
            state = jnp.moveaxis(state, 0, qubit)
        # flatten and get probabilities
        probs = jnp.abs(state.reshape(-1))**2
        idx   = jax.random.choice(key, probs.shape[0], p=probs)
        # bit‑decompose
        bits      = ((idx >> jnp.arange(N)[::-1]) & 1).astype(jnp.int32)
        outcomes  = 1 - 2*bits            # +1/–1
        return outcomes

    outcomes = jax.vmap(measure_one)(bases, keys)

    # encode bases as {2,3,4} and outcomes to {0,1}
    bases_enc = bases + 2
    out_enc   = ((outcomes + 1) // 2).astype(jnp.int32)

    # interleave to have the embedding [b0,o0,b1,o1,...]
    shadows = jnp.empty((num_shots, 2*N), dtype=jnp.int32)
    shadows = shadows.at[:, 0::2].set(bases_enc)
    shadows = shadows.at[:, 1::2].set(out_enc)
    return shadows



## Dataset class

The `Dataset` class centralizes quantum data, parameter grids, and help create training sets.
It supports:

- validating input tensor shapes and parameter grids
- creating shuffled training batches via `create_batches(...)`



In [ ]:
#| export


@dataclass
class Dataset:
    def __init__(self, data: jnp.ndarray, thetas: Sequence[jnp.ndarray], data_type: str, local_dimension: int=0, local_states: jnp.ndarray=None):
        """
        Dataset class for learning representation of quantum data with VAE: vae4q.
    
        Args:
        
            data: jnp.ndarray of shape (num_values_theta1, ..., num_values_thetak, num_sample_per_params, input_size)
            
            thetas: sequence of jnp.ndarray, each representing a parameter grid
            
            data_type: str, one of ['discrete', 'continuous', 'hybrid', 'shadow']
            
            local_dimension: int, dimension of the local Hilbert space
            
            local_states: jnp.ndarray of shape (local_dimension,) if data_type is 'discrete' specifying the local states possible values
        """
        
        
        #check the types
        if not isinstance(data, jnp.ndarray):
            raise TypeError(f"data must be a jax array, got {type(data)}")
        if not isinstance(thetas, list):
            raise TypeError(f"thetas must be a list, got {type(thetas)}")
        if not isinstance(data_type, str):
            raise TypeError(f"type must be a string, got {type(data_type)}")


        #check the sizes
        # if there are k thetas
        k = len(thetas)
        #data should have shape:(num_values_theta1, ..., num_values_thetak, num_sample_per_params, input_size)
        if data.ndim != k + 2:
            raise ValueError(f"Data should have {k+2} dimensions, got {data.ndim}")

        for i in range(k):
            if not isinstance(thetas[i], jnp.ndarray):
              raise TypeError(f"each thetas must be a jax array, got {type(thetas[i])}")
            if data.shape[i] != thetas[i].size:
                raise ValueError(f"data should have shape:(num_values_theta1, ..., num_values_thetak, num_sample_per_params, input_size)")

        #data type availables
        available_data_types = ['discrete', 'continuous', 'hybrid', 'shadow']
        if data_type not in available_data_types:
            raise ValueError(f"data_type must be one of {available_data_types}, got {data_type}")

        if data_type == 'discrete':
          if local_dimension == 0:
            raise ValueError(f"local_dimension must be specified for discrete data")


        if data_type != 'continuous' and jnp.size(local_states) != local_dimension:
          raise ValueError(f"local_states should have shape:(local_dimension,) got local_states={local_states.shape} and local_dimension={local_dimension}")

        self.data = data
        self.thetas = thetas
        self.data_type = data_type
        self.num_sample_per_params = data.shape[-2]
        self.input_size = data.shape[-1]
        self.local_dimension = local_dimension
        self.local_states = local_states




    def create_batches(self, batch_size: int, key: jnp.ndarray) -> list[jnp.ndarray]:
        """ create batches of data, shuffled"""
        dataset = self.data.reshape(-1,self.input_size)
        num_samples = dataset.shape[0]
        # Create a permutation of indices
        permutation = jax.random.permutation(key, jnp.arange(num_samples))
        # Shuffle the dataset
        shuffled_dataset = dataset[permutation]
        # Create batches from the shuffled dataset
        num_batches = num_samples // batch_size
        batches = []
        for i in range(num_batches):
            batch = shuffled_dataset[i * batch_size : (i + 1) * batch_size]
            batches.append(batch)
        return batches


In [ ]:
show_doc(Dataset.create_batches)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()